# Portfolio Optimization & Risk Management

**Comprehensive implementation of Phase 1-6 enhancements from portfolio_optimization_enhancement_plan.md**

This notebook integrates:
1. Advanced stock selection (Phase 1)
2. ML-based return prediction (Phase 2)
3. Sophisticated optimization methods (Phase 3)
4. Comprehensive risk management (Phase 4)
5. Robust backtesting framework (Phase 5)
6. Enhanced interactive dashboards (Phase 6)


In [ ]:
# Initial Setup and Imports
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

print("✅ Initial imports complete")


In [ ]:
# Data Loading
print("📊 Loading portfolio candidates from predictions.csv...")

predictions_path = Path("outputs/analytics/predictions.csv")
if not predictions_path.exists():
    raise FileNotFoundError(f"Predictions file not found: {predictions_path}")

portfolio_candidates = pd.read_csv(predictions_path)
print(f"✓ Loaded {len(portfolio_candidates)} stocks")
print(f"✓ Columns: {list(portfolio_candidates.columns[:10])}...")
portfolio_candidates.head()


In [ ]:
# Derive ranking metrics required by select_portfolio_candidates

# 1) Expected return (forward-looking) from mispricing
if "expected_return" not in portfolio_candidates.columns:
    if "mispricing_score" in portfolio_candidates.columns:
        # mispricing_score is already in decimal (e.g., 0.22 = 22% upside)
        portfolio_candidates["expected_return"] = portfolio_candidates["mispricing_score"].astype(float)
    elif "mispricing_pct" in portfolio_candidates.columns:
        portfolio_candidates["expected_return"] = (
                portfolio_candidates["mispricing_pct"].astype(float) / 100.0
        )
    elif {"predicted_price_target", "last_price"}.issubset(portfolio_candidates.columns):
        portfolio_candidates["expected_return"] = (
                portfolio_candidates["predicted_price_target"].astype(float)
                / portfolio_candidates["last_price"].astype(float)
                - 1.0
        )
    else:
        # Neutral expected return if nothing else is available
        portfolio_candidates["expected_return"] = 0.0

# 2) 1‑year historical return
if "return_1y" not in portfolio_candidates.columns:
    if "total_return_1y_pct" in portfolio_candidates.columns:
        portfolio_candidates["return_1y"] = (
                portfolio_candidates["total_return_1y_pct"].astype(float) / 100.0
        )
    elif "price_momentum_1y" in portfolio_candidates.columns:
        # price_momentum_1y is already in percent in the feature engine
        portfolio_candidates["return_1y"] = (
                portfolio_candidates["price_momentum_1y"].astype(float) / 100.0
        )
    else:
        # If no historical 1Y return is available, fall back to 0
        portfolio_candidates["return_1y"] = 0.0

print("✓ Derived ranking metrics: expected_return and return_1y")

## 10.1 Stock Selection (Phase 1)

Using `select_portfolio_candidates` from `analytics.stock_selection`


In [ ]:
from finance_ml.ml_workflow.analytics.stock_selection import (
    select_portfolio_candidates,
    rank_stocks_multi_metric,
    rank_stocks_balanced
    )

print("\n🎯 Phase 1: Enhanced Stock Selection")
print("=" * 60)

# Select top candidates with sector balance
selected_stocks = select_portfolio_candidates(
        df=portfolio_candidates,
        min_market_cap=5.0,  # 5B minimum
        top_n=30,
        max_sector_weight=0.30,
        cap_unit="B"
        )

print(f"\n✓ Selected {len(selected_stocks)} stocks")
print(f"\n📊 Sector distribution:")
print(selected_stocks['sector'].value_counts())

# Demonstrate additional ranking functions
print("\n📊 Multi-metric ranking (top 10):")
multi_ranked = rank_stocks_multi_metric(
        df=portfolio_candidates,
        metrics=['market_cap', 'mispricing_score'],
        weights=[0.3, 0.7],
        descending=True
        )
print(multi_ranked[['ticker', 'sector', 'market_cap', 'mispricing_score']].head(10))

print("\n📊 Balanced ranking across sectors:")
balanced_ranked = rank_stocks_balanced(
        df=portfolio_candidates,
        top_n=30,
        max_sector_weight=0.30,
        ranking_col='mispricing_score',
        sector_col='sector'
        )
print(f"✓ Selected {len(balanced_ranked)} stocks balanced across sectors")
print(balanced_ranked.groupby('sector').size())

selected_stocks.head(10)


## 10.2 ML-Based Return Prediction (Phase 2)

Using functions from `analytics.ml_returns`


In [ ]:
from finance_ml.ml_workflow.analytics.ml_returns import (
    create_ml_return_features,
    train_linear_return_predictor,
    create_ensemble_return_predictions,
    evaluate_return_predictions
    )

print("\n🤖 Phase 2: ML-Based Return Prediction")
print("=" * 60)

# Feature engineering
ml_features_df = create_ml_return_features(
        df=selected_stocks,
        lags=[1, 5, 21],
        technical_indicators=['momentum', 'volatility']
        )

print(f"✓ Created {len(ml_features_df.columns)} features")

# Prepare training data
feature_cols = [c for c in ml_features_df.columns if
                c.startswith('lag_') or c.startswith('momentum') or c.startswith('volatility')]
X = ml_features_df[feature_cols].fillna(0).values
y = ml_features_df['expected_return'].fillna(
    0).values if 'expected_return' in ml_features_df.columns else np.random.randn(len(X)) * 0.1

# Train model
model = train_linear_return_predictor(X, y)
print(f"\n✓ Model trained (alpha={model.alpha:.4f})")

# Generate predictions
ml_predictions = model.predict(X)
selected_stocks['ml_return_pred'] = ml_predictions

print(f"✓ ML predictions: mean={ml_predictions.mean():.4f}, std={ml_predictions.std():.4f}")

# Demonstrate ensemble predictions
print("\n📊 Creating ensemble predictions:")
# First add the ml_return_pred to create multiple prediction columns
selected_stocks['pred_ridge'] = ml_predictions
selected_stocks['pred_lasso'] = ml_predictions * 0.95  # Simulate slightly different predictions

ensemble_df = create_ensemble_return_predictions(
        df=selected_stocks,
        models=['pred_ridge', 'pred_lasso'],
        weights=[0.6, 0.4],
        ensemble_col='ensemble_return'
        )
print(f"✓ Ensemble predictions created")
print(f"  Mean: {ensemble_df['ensemble_return'].mean():.4f}, Std: {ensemble_df['ensemble_return'].std():.4f}")

# Evaluate predictions if we have true values
if 'expected_return' in selected_stocks.columns:
    print("\n📊 Evaluating return predictions:")
    y_true = selected_stocks['expected_return'].fillna(0).values
    eval_metrics = evaluate_return_predictions(
            y_true=y_true,
            y_pred=ml_predictions
            )
    print(f"✓ Correlation: {eval_metrics['correlation']:.4f}")
    print(f"  MAE: {eval_metrics['mae']:.4f}, RMSE: {eval_metrics['rmse']:.4f}")


## 10.3 Advanced Portfolio Optimization (Phase 3)

Black-Litterman, Risk Parity, and HRP optimization


In [ ]:
from finance_ml.ml_workflow.analytics.portfolio import (
    optimize_black_litterman,
    optimize_risk_parity,
    optimize_hrp,
    optimize_portfolio_max_sharpe
    )

print("\n🎯 Phase 3: Advanced Portfolio Optimization")
print("=" * 60)

# Prepare returns and covariance
expected_returns = selected_stocks['ml_return_pred'].values
n_stocks = len(selected_stocks)

# Synthetic covariance matrix
volatilities = np.abs(np.random.randn(n_stocks) * 0.15 + 0.20)
correlation = np.eye(n_stocks) + np.random.randn(n_stocks, n_stocks) * 0.1
correlation = (correlation + correlation.T) / 2
np.fill_diagonal(correlation, 1.0)
cov_matrix = np.outer(volatilities, volatilities) * correlation

# 1. Black-Litterman
print("\n📊 1. Black-Litterman Optimization")
market_weights = np.ones(n_stocks) / n_stocks
# Use ticker names as view keys (string keys as expected by type signature)
ticker_list = selected_stocks['ticker'].tolist() if 'ticker' in selected_stocks.columns else [f"Stock_{i}" for i in
                                                                                              range(n_stocks)]
views = {ticker_list[0]: 0.10, ticker_list[1]: 0.08}  # Views on first two stocks
view_confidences = [0.5, 0.5]

bl_result = optimize_black_litterman(
        returns=expected_returns,
        cov_matrix=cov_matrix,
        market_weights=market_weights,
        views=views,
        view_confidences=view_confidences
        )
bl_return = float(bl_result['return']) if isinstance(bl_result['return'], np.ndarray) else bl_result['return']
bl_vol = float(bl_result['volatility']) if isinstance(bl_result['volatility'], np.ndarray) else bl_result['volatility']
bl_weights = bl_result['weights'] if isinstance(bl_result['weights'], np.ndarray) else np.array(bl_result['weights'])
print(f"✓ BL Return: {bl_return:.2%}, Volatility: {bl_vol:.2%}")
print(f"  Top 3 weights: {sorted(bl_weights, reverse=True)[:3]}")

# 2. Risk Parity
print("\n📊 2. Risk Parity Optimization")
rp_result = optimize_risk_parity(cov_matrix=cov_matrix)
print(f"✓ RP Volatility: {rp_result['volatility']:.2%}")
print(f"  Weight range: [{rp_result['weights'].min():.3f}, {rp_result['weights'].max():.3f}]")

# 3. HRP (Hierarchical Risk Parity)
print("\n📊 3. Hierarchical Risk Parity (HRP)")
synthetic_returns = pd.DataFrame(np.random.multivariate_normal(expected_returns / 252, cov_matrix / 252, 252))
hrp_result = optimize_hrp(returns=synthetic_returns)
print(f"✓ HRP weights computed, sum={hrp_result['weights'].sum():.3f}")
print(f"  Top 3 weights: {sorted(hrp_result['weights'], reverse=True)[:3]}")

# 4. Maximum Sharpe Ratio (comparison)
print("\n📊 4. Maximum Sharpe Ratio Optimization")
max_sharpe_result = optimize_portfolio_max_sharpe(
        returns=expected_returns,
        cov_matrix=cov_matrix,
        risk_free_rate=0.03,
        allow_short=False,
        max_weight=0.15
        )
print(f"✓ Max Sharpe Return: {max_sharpe_result['return']:.2%}, Volatility: {max_sharpe_result['volatility']:.2%}")
print(f"  Sharpe Ratio: {max_sharpe_result['sharpe_ratio']:.3f}")
print(f"  Top 3 weights: {sorted(max_sharpe_result['weights'], reverse=True)[:3]}")


## 10.4 Risk Analysis (Phase 4)

Stress testing and Monte Carlo simulation


In [ ]:
from finance_ml.ml_workflow.analytics.risk import (
    calculate_expected_shortfall,
    calculate_tracking_error,
    run_stress_tests,
    run_monte_carlo_simulation
    )

print("\n⚠️ Phase 4: Risk Analysis")
print("=" * 60)

# Use Black-Litterman portfolio for risk analysis
portfolio_weights = bl_weights

# Generate synthetic daily returns for analysis
daily_returns = pd.DataFrame(np.random.multivariate_normal(
        expected_returns / 252,
        cov_matrix / 252,
        252
        ))

portfolio_returns = pd.Series((daily_returns.values @ portfolio_weights))

# 1. Expected Shortfall (CVaR)
print("\n📊 1. Expected Shortfall (CVaR)")
es_95 = calculate_expected_shortfall(portfolio_returns, confidence=0.95)
print(f"✓ CVaR (95%): {es_95:.4f}")

# 2. Tracking Error
print("\n📊 2. Tracking Error")
benchmark_returns = pd.Series(np.random.randn(252) * 0.01)
te = calculate_tracking_error(portfolio_returns, benchmark_returns)
print(f"✓ Tracking Error: {te:.4f}")

# 3. Stress Testing
print("\n📊 3. Stress Test Scenarios")
scenarios = {
    "Market Crash": {"equity": -0.20, "bond": 0.05},
    "Inflation Spike": {"equity": -0.10, "bond": -0.15},
    "Bull Market": {"equity": 0.25, "bond": 0.02}
    }
asset_classes = ['equity'] * n_stocks

stress_results = run_stress_tests(
        weights=portfolio_weights,
        returns=daily_returns,
        scenarios=scenarios,
        asset_class_mapping=asset_classes
        )
for scenario, impact in stress_results.items():
    print(f"  {scenario}: {impact:.2%}")

# 4. Monte Carlo Simulation
print("\n📊 4. Monte Carlo Simulation")
mc_results = run_monte_carlo_simulation(
        weights=portfolio_weights,
        returns=daily_returns,
        n_simulations=10000,
        time_horizon=252,
        confidence_levels=[0.05, 0.50, 0.95]
        )
print(f"✓ Simulated outcomes:")
print(f"  5th percentile: {mc_results['percentiles'][0.05]:.2%}")
print(f"  Median: {mc_results['percentiles'][0.50]:.2%}")
print(f"  95th percentile: {mc_results['percentiles'][0.95]:.2%}")


## 10.5 Backtesting Framework (Phase 5)

Vectorized backtest and performance attribution


In [ ]:
from finance_ml.ml_workflow.analytics.portfolio import (
    run_vectorized_backtest,
    run_walk_forward_optimization
    )
from finance_ml.ml_workflow.analytics.attribution import (
    calculate_performance_attribution
    )

print("\n📈 Phase 5: Backtesting Framework")
print("=" * 60)

# Generate synthetic historical data
dates = pd.date_range('2020-01-01', periods=756, freq='D')
historical_prices = pd.DataFrame(
        100 * np.exp(np.random.randn(756, n_stocks).cumsum(axis=0) * 0.01),
        index=dates,
        columns=[f"Asset_{i}" for i in range(n_stocks)]
        )

# 1. Vectorized Backtest
print("\n📊 1. Vectorized Backtest")
backtest_results = run_vectorized_backtest(
        data=historical_prices,
        rebalance_frequency="monthly",
        optimization_method="max_sharpe",
        lookback_window=252,
        transaction_costs=0.001
        )
print(f"✓ Portfolio Value: ${backtest_results['portfolio_value'][-1]:,.2f}")
print(f"✓ Total Return: {(backtest_results['portfolio_value'][-1] / backtest_results['portfolio_value'][0] - 1):.2%}")
print(f"✓ Sharpe Ratio: {backtest_results['sharpe_ratio']:.3f}")

# 2. Walk-Forward Optimization
print("\n📊 2. Walk-Forward Optimization")
wfo_results = run_walk_forward_optimization(
        data=historical_prices,
        train_window=252,
        test_window=63,
        step_size=21,
        optimization_method="black_litterman"
        )
print(f"✓ Test windows: {len(wfo_results['test_returns'])}")
print(f"✓ Average test return: {np.mean(wfo_results['test_returns']):.4f}")

# 3. Performance Attribution
print("\n📊 3. Performance Attribution (Brinson-Fachler)")
# Create sample sector-level data
sectors = ['Tech', 'Finance', 'Healthcare']
portfolio_weights_df = pd.DataFrame([[0.5, 0.3, 0.2]], columns=sectors)
benchmark_weights_df = pd.DataFrame([[0.4, 0.4, 0.2]], columns=sectors)
portfolio_returns_df = pd.DataFrame([[0.10, 0.05, 0.08]], columns=sectors)
benchmark_returns_df = pd.DataFrame([[0.08, 0.06, 0.07]], columns=sectors)

attribution = calculate_performance_attribution(
        portfolio_weights=portfolio_weights_df,
        portfolio_returns=portfolio_returns_df,
        benchmark_weights=benchmark_weights_df,
        benchmark_returns=benchmark_returns_df
        )
print(f"✓ Allocation Effect: {attribution['allocation_effect']:.4f}")
print(f"✓ Selection Effect: {attribution['selection_effect']:.4f}")
print(f"✓ Interaction Effect: {attribution['interaction_effect']:.4f}")
print(f"✓ Total Active Return: {sum(attribution.values()):.4f}")


## 10.6 Interactive Dashboard (Phase 6)

Portfolio rebalancing widget and multi-period visualizations


In [ ]:
from finance_ml.dashboards.portfolio_widgets import (
    PortfolioRebalanceWidget,
    create_multi_period_comparison,
    create_factor_exposure_dashboard
    )

print("\n🎨 Phase 6: Interactive Dashboard Components")
print("=" * 60)

# 1. Portfolio Rebalancing Widget
print("\n📊 1. Portfolio Rebalancing Widget")
current_holdings = pd.DataFrame({
    'ticker': selected_stocks['ticker'].head(10).values,
    'shares': np.random.randint(50, 200, 10),
    'price': selected_stocks['last_price'].head(10).values
    })

target_weights = pd.Series(
        bl_weights[:10],
        index=selected_stocks['ticker'].head(10).values
        )

widget = PortfolioRebalanceWidget(
        current_holdings=current_holdings,
        target_weights=target_weights
        )

trades = widget.get_rebalance_trades()
print(f"✓ Generated {len(trades)} rebalancing trades")
print(trades.head())

# Save as HTML
output_path = Path("outputs/analytics/portfolio_rebalance_widget.html")
output_path.parent.mkdir(parents=True, exist_ok=True)
trades.to_html(output_path)
print(f"✓ Saved to {output_path}")

# 2. Multi-Period Comparison
print("\n📊 2. Multi-Period Performance Comparison")
portfolio_daily_returns = pd.Series(
        np.random.randn(252) * 0.01 + 0.0005,
        index=pd.date_range('2024-01-01', periods=252, freq='D')
        )
benchmark_daily_returns = pd.Series(
        np.random.randn(252) * 0.008 + 0.0003,
        index=pd.date_range('2024-01-01', periods=252, freq='D')
        )

fig_comparison = create_multi_period_comparison(
        portfolio_returns=portfolio_daily_returns,
        periods=["1M", "3M", "6M", "1Y", "YTD"],
        benchmark_returns=benchmark_daily_returns
        )

output_path = Path("outputs/analytics/portfolio_multi_period_comparison.html")
fig_comparison.write_html(output_path)
print(f"✓ Saved to {output_path}")
fig_comparison.show()

# 3. Factor Exposure Dashboard
print("\n📊 3. Factor Exposure Dashboard")
portfolio_weights_series = pd.Series(
        bl_weights[:10],
        index=[f"Stock_{i}" for i in range(10)]
        )

factors = ["Market", "Size", "Value", "Momentum", "Quality"]
factor_loadings = pd.DataFrame(
        np.random.randn(10, 5) * 0.5,
        index=[f"Stock_{i}" for i in range(10)],
        columns=factors
        )

fig_factors = create_factor_exposure_dashboard(
        portfolio_weights=portfolio_weights_series,
        factor_loadings=factor_loadings,
        factors=factors
        )

output_path = Path("outputs/analytics/portfolio_factor_exposure_dashboard.html")
fig_factors.write_html(output_path)
print(f"✓ Saved to {output_path}")
fig_factors.show()

print("\n✅ Phase 6 Complete - All visualizations generated!")


## Summary

This notebook has successfully integrated all Phase 1-6 enhancements:

✅ **Phase 1**: Enhanced stock selection with sector balance  
✅ **Phase 2**: ML-based return prediction with ensemble methods  
✅ **Phase 3**: Advanced optimization (Black-Litterman, Risk Parity, HRP)  
✅ **Phase 4**: Comprehensive risk analysis (CVaR, stress tests, Monte Carlo)  
✅ **Phase 5**: Robust backtesting with performance attribution  
✅ **Phase 6**: Interactive dashboards and visualizations  

**Outputs Generated**:
- `outputs/analytics/portfolio_rebalance_widget.html`
- `outputs/analytics/portfolio_multi_period_comparison.html`
- `outputs/analytics/portfolio_factor_exposure_dashboard.html`
